In [ ]:
import sys
import time
import h5py
import torch
import math
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from utils import log_accuracy, run_tests

EXTEND_LABELS = False
USE_FFT = False
USE_FFT_STACKING = False
USE_CONVOLUTIONS = False
USE_BATCH_NORM = False

USE_EXPERIMENT_DATA = False

# Data loading

In [ ]:
# Connect torch to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

In [ ]:
training_data_file = '../data/experiment/sdr_wifi_train.hdf5' if USE_EXPERIMENT_DATA else '../data/sdr_wifi_train.hdf5'
test_data_file = '../data/experiment/sdr_wifi_test.hdf5' if USE_EXPERIMENT_DATA else '../data/sdr_wifi_test.hdf5'

# Load test set
f = h5py.File(test_data_file, 'r')
X_test = f['X'][()]
y_test = f['y'][()]
f.close()

# Load train set
f = h5py.File(training_data_file, 'r')
X_train = f['X'][()]
y_train = f['y'][()]
f.close()

In [ ]:
def change_label(y):
    """
    y: shape (N, 4), where each row is a one-hot or multi-hot vector for 4 classes.
    Returns: shape (N, 13), expanded labels.
    """
    N = y.shape[0]
    results = np.zeros((N, 13), dtype=np.float32)

    # Define the mapping for each class index
    mapping = {
        0: [0, 1, 2],
        1: [2, 3, 4, 5, 6],
        2: [6, 7, 8, 9, 10],
        3: [10, 11, 12]
    }

    for class_idx, indices in mapping.items():
        mask = y[:, class_idx] == 1
        for idx in indices:
            results[mask, idx] = 1

    return results

In [ ]:
if EXTEND_LABELS:
    y_train = change_label(y_train)
    y_test = change_label(y_test)

# Description of input variables

- `sequence_length` is the amount of IQ samples is in a single sample of data
- `nm_channels` is the amount of number each IQ samples consists of (2)
- `num_layers` is the amount of hidden layers that perform changes to get the correct prediction (chosen based on https://stats.stackexchange.com/questions/181/how-to-choose-the-number-of-hidden-layers-and-nodes-in-a-feedforward-neural-netw)
- `output_size` is the amount of frequencies that are predicted, these are equal to the input since we want to see from the full prediction which frequency most likely has the lowest interference
- `num_epochs` is the amount of training rounds
- `learning_rate` is the rate at which the weights of the hidden layers are updated to improve prediction results (cannot be too high because it might overshoot)


In [ ]:
data = torch.FloatTensor(X_train).to(device)
labels = torch.FloatTensor(y_train).to(device)

print(data.shape)
print(labels.shape)

# Neural network input
sequence_length = data.shape[1]
num_channels = data.shape[2]
num_layers = 1
output_size = labels.shape[1]

learning_rate = 1e-3
num_epochs = 30
batch_size = 128

# Model definition

In [ ]:
USE_LOGITS = True

In [ ]:
class SpectrumModel(nn.Module):
    def __init__(self, sequence_length, num_channels = 2, output_size = 4):
        super(SpectrumModel, self).__init__()
        if USE_CONVOLUTIONS:
            # Convolutional layers to extract more features from IQ samples improving classification
            self.conv1 = nn.Conv1d(in_channels=num_channels, out_channels=16, kernel_size=3, stride=1, padding=1)
            self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)

        if USE_BATCH_NORM:
            self.bn1 = nn.BatchNorm1d(16)
            self.bn2 = nn.BatchNorm1d(32)

        if USE_CONVOLUTIONS:
            # Fully connected layers to classify the signal with it's features extracted by the convolutional layers
            self.fc1 = nn.Linear(32 * sequence_length * 2 if USE_FFT_STACKING else 32 * sequence_length, 128)
        else:
            self.fc1 = nn.Linear(sequence_length * 2 * 2 if USE_FFT_STACKING else sequence_length * 2, 128)

        self.fc2 = nn.Linear(128, output_size)

        # Leaky relu
        self.activation = nn.LeakyReLU(negative_slope=0.01)

    def forward(self, x):
        if USE_FFT:
            # Optional Step 1: Use fft on data
            x = torch.fft.fft(x, dim=1) # -> [batch_size, sequence_length, 2]
            x = torch.abs(x) # -> [batch_size, sequence_length, 2]
        elif USE_FFT_STACKING:
            # Optional Step 1: Use fft on data and stack the real and imaginary parts
            fft_x = torch.fft.fft(x, dim=1)
            fft_x = torch.abs(fft_x)  # -> [batch_size, sequence_length, 2]
            # Stack time and frequency domain representations
            x = torch.cat((x, fft_x), dim=1)

        if USE_CONVOLUTIONS:
                # Step 1: get more information out of the IQ samples using convolution
            # Input shape: [batch_size, sequence_length, 2]
            x = x.permute(0, 2, 1)  # -> [batch_size, 2, sequence_length]

            # Step 1: First convolutional layer
            x = self.conv1(x) # -> [batch_size, 16, sequence_length]

            if USE_BATCH_NORM:
                x = self.bn1(x) # -> [batch_size, 16, sequence_length]

            x = torch.relu(x) # -> [batch_size, 16, sequence_length]

            # Step 2: Second convolutional layer
            x = self.conv2(x) # -> [batch_size, 32, sequence_length]

            if USE_BATCH_NORM:
                x = self.bn2(x) # -> [batch_size, 32, sequence_length]

            x = torch.relu(x) # -> [batch_size, 32, sequence_length]

        # Step 2: flatten the output of the convolutional layers
        x = x.view(x.size(0), -1)  # -> [batch_size, 32 * sequence_length] or [batch_size, 2 * sequence_length]

        # Step 3: use a fully connected layer to classify the signal
        x = self.fc1(x) # -> [batch_size, 128]
        x = self.activation(x) # -> [batch_size, 128]
        x = self.fc2(x) # -> [batch_size, output_size]

        if not USE_LOGITS:
            x = torch.sigmoid(x)
        
        return x

# Initialize the model, loss function, and optimizer
model = SpectrumModel(
    sequence_length=sequence_length,
    num_channels=num_channels,
    output_size=output_size
).to(device)
criterion = nn.BCEWithLogitsLoss() if USE_LOGITS else nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# Training

In [ ]:
dataset = TensorDataset(data, labels)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

avg_loss_history = []
avg_loss = 0

min_loss_history = []
max_loss_history = []

start_time = time.time()

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    min_loss = 10
    max_loss = 0

    for i, (x_batch, y_batch) in enumerate(train_loader):
        # Forward pass
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        total_loss += current_loss
        min_loss = min(min_loss, current_loss)
        max_loss = max(max_loss, current_loss)

        sys.stdout.write(f"\rEpoch [{epoch + 1}/{num_epochs}] | ({i} / {math.floor(len(data) / batch_size)}) | Loss: {loss.item():.4f} | Avg Loss last epoch: {avg_loss:.4f}")

    avg_loss = total_loss / len(train_loader)
    avg_loss_history.append(avg_loss)
    min_loss_history.append(min_loss)
    max_loss_history.append(max_loss)

    # Step the scheduler
    scheduler.step()

end_time = time.time()

training_time = end_time - start_time

# Testing and documentation

In [ ]:
def get_model_name():
    model_name = f"sdr_wifi_model_{int(time.time())}"
    if USE_FFT:
        model_name += "_fft"
    elif USE_FFT_STACKING:
        model_name += "_fft_stacking"
    if USE_CONVOLUTIONS:
        model_name += "_conv"
    if USE_BATCH_NORM:
        model_name += "_bn"
    return model_name

In [ ]:
test_results = run_tests(
    model=model,
    criterion=criterion,
    X_test=X_test,
    y_test=y_test,
    batch_size=batch_size,
    device=device,
)

In [ ]:
log_accuracy(
    model=get_model_name(),
    label=f'{get_model_name()} (recorded data)',
    data=data,
    training_time=training_time,
    num_epochs=num_epochs,
    min_loss=min_loss,
    max_loss=max_loss,
    avg_loss=test_results['avg_loss'],
    accuracy=test_results['accuracy'],
    avg_response_time=test_results['avg_response_time'],
    sampling_rate=1
)